# Etapa 2. Modelo de recomendación basado en descripción

Nuestro objetivo es crear un modelo que permita recomendar los videojuegos más parecidos al juego que le indiquemos, entregará estas recomendaciones basandose unicamente en las descripciones que hemos lematizado previamente y se encuentran en nuestro dataset preprocesado.

### 1. Inicialización

In [1]:
import pandas as pd
import numpy as np

from nltk.corpus import stopwords as nltk_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import save_npz
import pickle

### 2. Carga de datos

In [2]:
data = pd.read_csv('datasets/steam_games_preprocessed.csv', parse_dates=['release_date'])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   appid                27075 non-null  int64         
 1   name                 27075 non-null  object        
 2   release_date         27075 non-null  datetime64[ns]
 3   english              27075 non-null  int64         
 4   developer            27075 non-null  object        
 5   publisher            27075 non-null  object        
 6   platforms            27075 non-null  object        
 7   required_age         27075 non-null  int64         
 8   genres               27075 non-null  object        
 9   steamspy_tags        27075 non-null  object        
 10  positive_ratings     27075 non-null  int64         
 11  negative_ratings     27075 non-null  int64         
 12  price                27075 non-null  float64       
 13  description          27075 non-

In [3]:
data.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,genres,steamspy_tags,positive_ratings,negative_ratings,price,description,genres_lists,steamspy_tags_lists,description_lemm,year
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Action,"Action, FPS, Multiplayer",124534,3339,7.19,Play the world's number 1 online action game. ...,['Action'],"['Action', 'FPS', 'Multiplayer']",play world number online action game Engage in...,2000
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Action,"Action, FPS, Multiplayer",3318,633,3.99,One of the most popular online action games of...,['Action'],"['Action', 'FPS', 'Multiplayer']",one popular online action game time Team Fortr...,1999
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Action,"FPS, World War II, Multiplayer",3416,398,3.99,Enlist in an intense brand of Axis vs. Allied ...,['Action'],"['FPS', 'World War II', 'Multiplayer']",Enlist intense brand Axis vs Allied teamplay s...,2003
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Action,"Action, FPS, Multiplayer",1273,267,3.99,Enjoy fast-paced multiplayer gaming with Death...,['Action'],"['Action', 'FPS', 'Multiplayer']",enjoy fast pace multiplayer game Deathmatch Cl...,2001
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Action,"FPS, Action, Sci-fi",5250,288,3.99,Return to the Black Mesa Research Facility as ...,['Action'],"['FPS', 'Action', 'Sci-fi']",return Black Mesa Research Facility one milita...,1999


### 3. Reemplacemos nulos 

In [4]:
data['description_lemm'] = data['description_lemm'].fillna('')

### 4. Matriz de valores TF-IDF

In [5]:
stop_words = list(nltk_stopwords.words('english'))

count_tf_idf = TfidfVectorizer(stop_words=stop_words,
                              min_df=5,
                              max_df=0.7,
                              ngram_range=(1,2)
                             )
tf_idf_desc = count_tf_idf.fit_transform(data['description_lemm'])

In [6]:
tf_idf_desc.shape

(27075, 114808)

In [7]:
#Guardemos la matriz
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(count_tf_idf, f)

with open('models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tf_idf_desc, f)


### 5. Matriz de similitud Coseno

Nos permitira ver que tan similares son 2 juegos.

In [8]:
similarity_matrix_desc = cosine_similarity(tf_idf_desc)

In [9]:
#guardemos la matriz
with open('models/similarity_matrix.pkl', 'wb') as f:
    pickle.dump(similarity_matrix_desc, f)

### 6. Función de recomendación

Finalmente crearemos nuestra función de recomendación, la cual al ingresar un nombre de videojuego, si este se encuentra en nuestro dataframe, nos devolverá los videojuegos más parecidos a este.

In [10]:
def recommend_games(game_name, data, similarity_matrix, top_n=5):
    if game_name not in data['name'].values:
        raise ValueError('No encontrado en la base de datos')

    game_index =  data.index[data['name'] == game_name][0]

    sim_scores = list(enumerate(similarity_matrix[game_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    recommended_games = data.iloc[[index for index, _ in sim_scores]][['appid', 'name', 'genres', 'steamspy_tags']]

    return recommended_games

In [11]:
#prueba
print(data[data['name'] == 'Counter-Strike'][['name', 'genres', 'steamspy_tags']])
recommend_games('Counter-Strike', data, similarity_matrix_desc)

             name  genres             steamspy_tags
0  Counter-Strike  Action  Action, FPS, Multiplayer


,appid,name,genres,steamspy_tags
13031,584780,Operation swat,"Violent, Action, Strategy","Action, Violent, Strategy"
19691,781290,Hostage: Rescue Mission,"Action, Strategy","Strategy, Action"
1,20,Team Fortress Classic,Action,"Action, FPS, Multiplayer"
23047,881060,Fortune & Gloria,"Action, Adventure, Casual, Indie","Action, Indie, Adventure"
24647,939000,Crisis VRigade,"Action, Casual, Indie, Early Access","Early Access, Action, Indie"


In [12]:
#Guardemos nuestra función
with open('models/recommender_function.pkl', 'wb') as f:
    pickle.dump(recommend_games, f)


### 7. Conclusiones

1. Hemos encontrado los valores TF-IDF a nuestras descripciones previamente lematizadas.
2. Generamos nuestra matriz de similitud.
3. Creamos nuestra función que recomienda los 5 videojuegos más parecidos al indicado.
4. Guardamos nuestra información para su posterior uso en Streamlit y en la construcción del modelo hibrido.